# K-Means Clustering

In [ ]:
# Remember: library imports are ALWAYS at the top of the script, no exceptions!
import sqlite3
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.impute import KNNImputer
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder
from math import ceil

from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram

## New imports
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.cluster import KMeans


sns.set()

In [ ]:
## Un-comment these if you want to use ydata_profiling

# !pip install -U ydata-profiling
# from ydata_profiling import ProfileReport


## Context
The data we will be using through the pratical classes comes from a small relational database whose schema can be seen below:

![Schema](https://raw.githubusercontent.com/fpontejos/DMDM_2223/main/figures/schema.png "Relation database schema")

## Reading the Data

In [ ]:
## Allow Colab to see Google Drive files

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
## Load csv file into a dataframe
## Paste the path here
data_path = "/content/drive/MyDrive/Colab Data/datamining.csv"

df = pd.read_csv(data_path)

In [ ]:
## Load csv file into a dataframe
## Alternative location of the csv file

## data_path = "https://raw.githubusercontent.com/fpontejos/DM1_2324/main/data/datamining.csv"

## df = pd.read_csv(data_path)

### Make a copy of your original dataset

why?

In [ ]:
df_original = df.copy()

### Metadata
- *id* - The unique identifier of the customer
- *age* - The year of birht of the customer
- *income* - The income of the customer
- *frq* - Frequency: number of purchases made by the customer
- *rcn* - Recency: number of days since last customer purchase
- *mnt* - Monetary: amount of € spent by the customer in purchases
- *clothes* - Number of clothes items purchased by the customer
- *kitchen* - Number of kitchen items purchased by the customer
- *small_appliances* - Number of small_appliances items purchased by the customer
- *toys* - Number of toys items purchased by the customer
- *house_keeping* - Number of house_keeping items purchased by the customer
- *dependents* - Binary. Whether or not the customer has dependents
- *per_net_purchase* - Percentage of purchases made online
- *education* - Education level of the customer
- *status* - Marital status of the customer
- *gender* - Gender of the customer
- *description* - Last customer's recommendation description

## Preprocess the Data

Remember what we did in the previous session

In [ ]:
# Sometimes it is not obvious that a value is missing
# For example if the value is an empty string

# replace "" by nans
df.replace("", np.nan, inplace=True)

In [ ]:
df["dependents"] = df["dependents"].astype("boolean")


In [ ]:
# Define metric and non-metric features. Why?
non_metric_features = ["education", "status", "gender", "dependents", "description"]

## This is saying that the metric features are all the other features that are not non-metric
## Need to be careful in case not all columns are to be used as features

# metric_features = df.columns.drop(non_metric_features).to_list()

## Or you can also specify manually
metric_features = ['age',
 'income',
 'frq',
 'rcn',
 'mnt',
 'clothes',
 'kitchen',
 'small_appliances',
 'toys',
 'house_keeping',
 'per_net_purchase']

### Fill missing values (Data imputation)

How can we fill missing values?


#### Using measures of central tendency

In [ ]:
# Creating a copy to apply central tendency measures imputation
df_central = df.copy()

In [ ]:
# count of missing values
df_central.isna().sum()

In [ ]:
medians = df_central[metric_features].median()

In [ ]:
modes = df_central[non_metric_features].mode().loc[0]

In [ ]:
## Fill NaNs using medians and modes

df_central.fillna(medians, inplace=True)
df_central.fillna(modes, inplace=True)

In [ ]:
df_central.isna().sum()  # checking how many NaNs we still have

In [ ]:
df = df_central.copy()

### Outlier removal

In [ ]:
def remove_outliers(df, filters):

  df_2 = df[filters]
  print('Percentage of data kept after removing outliers:', 100*(np.round(df_2.shape[0] / df.shape[0], 4)))

  return df_2



In [ ]:
# This may vary from session to session, and is prone to varying interpretations.
# A simple example is provided below:

manual_filters = (
    (df['house_keeping']<=50)
    &
    (df['kitchen']<=40)
    &
    (df['toys']<=35)
    &
    (df['education']!='OldSchool')
)

df_1 = remove_outliers(df, manual_filters)

In [ ]:
# Get the manual filtering version
df = df_1.copy()

In [ ]:
# How can we avoid having as many extreme values in 'rcn'?
print((df['rcn']>100).value_counts())

rcn_t = df['rcn'].copy()
rcn_t.loc[rcn_t>100] = 100

df['rcn'] = rcn_t

#### What about non-metric features?

In [ ]:
# Let's also remove status=Whatever
df.loc[df['status'] == 'Whatever', 'status'] = df['status'].mode()[0]


### Feature Engineering and Feature Selection

In [ ]:
df['birth_year'] = df['age']
df['age'] = datetime.now().year - df['birth_year']

df['spent_online'] = (df['per_net_purchase'] / 100) * df['mnt']

#### Redundancy


In [ ]:
# Select variables according to their correlations
df.drop(columns=['birth_year', 'age', 'mnt'], inplace=True)

In [ ]:
# Updating metric_features
metric_features.append("spent_online")
metric_features.remove("mnt")
metric_features.remove("age")

In [ ]:
metric_features

#### Relevancy
Selecting variables based on the relevancy of each one to the task. Example: remove uncorrelated variables with the target, stepwise regression, use variables for product clustering, use variables for socio-demographic clustering, ...

Variables that aren't correlated with any other variable are often also not relevant. In this case we will not focus on this a lot since we don't have a defined task yet.

### Data Normalization

#### Standard Scaling

In [ ]:
df_standard = df.copy()

In [ ]:
scaler = StandardScaler()
scaled_feat = scaler.fit_transform(df_standard[metric_features])
df_standard[metric_features] = scaled_feat


In [ ]:
df = df_standard.copy()

### One-hot encoding

In [ ]:
df_ohc = df.copy()

In [ ]:
def get_ohc_df(df, feats):
  # Use OneHotEncoder to encode the categorical features.
  # Get feature names and create a DataFrame
  # with the one-hot encoded categorical features (pass feature names)

  ohc = OneHotEncoder(sparse_output=False, drop="first")
  ohc_feat = ohc.fit_transform(df[feats])
  ohc_feat_names = ohc.get_feature_names_out()
  ohc_df = pd.DataFrame(ohc_feat, index=df.index, columns=ohc_feat_names)

  # Reassigning df to contain ohc variables
  df_ohc = pd.concat([df, ohc_df], axis=1)

  ## Return the df with the one-hot encoded features
  ## Also return the OneHotEncoder model (ohc)
  return df_ohc, ohc

df_ohc, ohc = get_ohc_df(df, non_metric_features)


In [ ]:
oh_features = ohc.get_feature_names_out().tolist()
oh_features

In [ ]:
df_ohc.columns

In [ ]:
df = df_ohc.copy()

### OR... Import preprocessed data


## K-Means Clustering
What is K-Means clustering? How does it work?

https://www.youtube.com/watch?v=5I3Ei69I40s


### How is it computed?

![](https://raw.githubusercontent.com/fpontejos/DM1_2324/main/figures/kmeans.png)

### Characteristics:
- *Number of clusters* need to be set apriori
- One of the *fastest* clustering algorithms
- The results *depend on the initialization* (stochastic)
- Prone to *local optima*
- Favors *convex* (round shape) and *isotropic* (same shape) clusters

### How to apply K-Means clustering?

In [ ]:
## What kinds of features can we use?

kmclust = KMeans(n_clusters=8, init='random', n_init=10, random_state=1)

# the fit method
kmclust.fit(df[metric_features])

In [ ]:
# The predict method
# This gives you the cluster label of each row
kmclust.predict(df[metric_features])

In [ ]:
# The transform method
# This gives you the distances of each row to each of the cluster centroids

pd.DataFrame(kmclust.transform(df[metric_features]))

### How can we improve the initialization step?

https://www.youtube.com/watch?v=9nKfViAfajY


In [ ]:
# Better initialization method and provide more n_init
kmclust = KMeans(n_clusters=8,
                 init='k-means++',  ## notice different initialization algorithm
                 n_init=15,         ## notice different value
                 random_state=1)    ## why set random_state?

kmclust.fit(df[metric_features])
kmclust.predict(df[metric_features])


*init='k-means++'* initializes the centroids to be (generally) distant from each other, leading to probably better results than random initialization. *n_init=K* allows to initialize KMeans K times and pick the best clustering in terms of Inertia. This can been shown in the link below.

**Empirical evaluation of the impact of k-means initialization:**

https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_stability_low_dim_dense.html#sphx-glr-auto-examples-cluster-plot-kmeans-stability-low-dim-dense-py

### How do we find the number of clusters?

**Inertia (within-cluster sum-of-squares distance) Formula:**
$$\sum_{j=0}^{C}\sum_{i=0}^{n_j}(||x_i - \mu_j||^2)$$
, where:

$C$: Set of identified clusters.

$n_j$: Set of observations belonging to cluster $j$.

$x_i$: Observation $i$.

$\mu_j$: Centroid of cluster $j$.

---

*remember SS_w?*

![](https://raw.githubusercontent.com/fpontejos/DM1_2324/main/figures/ssw_ssb.png)

In [ ]:
def plot_inertia(df, feats, max_k=10):
  range_clusters = range(1, max_k+1)

  inertia = []
  for n_clus in range_clusters:  # iterate over desired ncluster range
      kmclust = KMeans(n_clusters=n_clus, init='k-means++', n_init=15, random_state=1)
      kmclust.fit(df[feats])
      inertia.append(kmclust.inertia_)  # save the inertia of the given cluster solution


  # The inertia plot
  plt.figure(figsize=(9,5))
  plt.plot(range_clusters, inertia)
  plt.ylabel("Inertia: SSw")
  plt.xlabel("Number of clusters")
  plt.xticks(range_clusters)
  plt.title("Inertia plot over clusters", size=15)
  plt.show()

In [ ]:
plot_inertia(df, metric_features)

---

**Silhouette Coefficient formula for a single sample:**
$$s = \frac{b - a}{max(a, b)}$$
, where:
- $a$: The mean distance between a sample and all other points in the same cluster.
- $b$: The mean distance between a sample and all other points in the next nearest cluster

---


If $b > a$, then what?

Then the sample is closer to the points in the cluster it is assigned to (compared to the points in the next nearest cluster)

$s$ is positive

---


If $b = a$, then what?

*Then the sample is equally distant to the points in the cluster it is assigned to as well as the points in the next closest cluster*

$s$ is 0

---

If $b < a$, then what?

*Then the sample is closer to the points in the next closest cluster (compared to the points in the same cluster). *

$s$ is negative

---

If the average value of $s$ is high, then what?

**


"Silhouette coefficients (as these values are referred to as) near +1 indicate that the sample is far away from the neighboring clusters. A value of 0 indicates that the sample is on or very close to the decision boundary between two neighboring clusters and negative values indicate that those samples might have been assigned to the wrong cluster."

- https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_silhouette_analysis.html





In [ ]:
# Adapted from:
# https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_silhouette_analysis.html#sphx-glr-auto-examples-cluster-plot-kmeans-silhouette-analysis-py

def plot_silhouette_score(df, feats, max_k=10):
  range_clusters = range(2, max_k+1)
  # Skip nclus == 1

  # Storing average silhouette metric
  avg_silhouette = []
  for nclus in range_clusters:
      # Initialize the KMeans object with n_clusters value
      # and a random generator seed for reproducibility.
      kmclust = KMeans(n_clusters=nclus, init='k-means++', n_init=15, random_state=1)
      cluster_labels = kmclust.fit_predict(df[feats])

      # The silhouette_score gives the average value for all the samples.
      # This gives a perspective into the density and separation of the formed clusters
      silhouette_avg = silhouette_score(df[feats], cluster_labels)
      avg_silhouette.append(silhouette_avg)
      print(f"For n_clusters = {nclus}, the average silhouette_score is : {silhouette_avg}")

  # The average silhouette plot
  plt.figure(figsize=(9,5))
  plt.plot(range_clusters, avg_silhouette)
  plt.ylabel("Average silhouette")
  plt.xlabel("Number of clusters")
  plt.xticks(range_clusters)
  plt.title("Average silhouette plot over clusters", size=15)
  plt.show()


In [ ]:
plot_silhouette_score(df, metric_features)

### Final KMeans clustering solution

In [ ]:
# final cluster solution
number_clusters = 3
kmclust = KMeans(n_clusters=number_clusters, init='k-means++', n_init=15, random_state=1)
km_labels = kmclust.fit_predict(df[metric_features])
km_labels

In [ ]:
df_km_labeled = pd.concat((df, pd.Series(km_labels, name='km_labels', index=df.index)),
                        axis=1)
df_km_labeled

In [ ]:
## Characterize the clusters
## Just like we did for Hierarchical Clustering

def get_mean_bylabel(df, feats, label_name):
  # Characterizing the clusters
  return df[feats+[label_name]].groupby(label_name).mean()


In [ ]:
get_mean_bylabel(df_km_labeled, metric_features, 'km_labels')

### How can we combine the 2 algorithms (K-Means and Hierarchical)?

## Next: DBSCAN and Clustering by Perspectives

### Questions?

## [OPTIONAL] Exercise

To practice your clustering skills, you can do the same exercises on a different dataset.


The Spaceship Titanic Dataset has been loaded for you in the cells below. You can find more information about this dataset from the Kaggle link.


Using this notebook as a guide, try to answer the questions that follow.


---

Addison Howard, Ashley Chow, Ryan Holbrook. (2022). Spaceship Titanic. Kaggle. https://kaggle.com/competitions/spaceship-titanic


In [ ]:
titanic_df = pd.read_csv("https://raw.githubusercontent.com/fpontejos/DM1_2324/main/data/spaceship_titanic_dataset.csv")


In [ ]:
## Keep only the useful features that you identified in the previous exercise

titanic_metric_features = []
titanic_non_metric_features = []


In [ ]:
## Don't forget to do the preprocessing steps you performed in the previous exercise


In [ ]:
## Perform the KMeans Clustering algorithm on your data
## making sure to use the appropriate hyperparameters such as number of clusters
